# Queue Statistics – Plots

In [ ]:
import stata_setup
stata_setup.config('/Applications/StataNow', 'mp')

ModuleNotFoundError: No module named 'environ'

In [ ]:
%%stata

clear all
set more off

global PROCESSED_DATA "../../processed_data"
global FIGURES        "../../figures"

In [ ]:
%%stata
****************************
* Queue Length Over Time
****************************

import delimited using "$PROCESSED_DATA/queue_daily.csv", varnames(1) clear
gen date2 = date(date, "YMD")
format date2 %td

twoway ///
    (line queue_length date2, lcolor(navy) lwidth(medthick)) ///
    , ///
    xtitle("") ///
    ytitle("Number of Requests") ///
    title("Lido Withdrawal Queue Length") ///
    xlabel(, format(%tdMon-CCYY) angle(45)) ///
    graphregion(color(white)) ///
    plotregion(margin(zero))

graph export "$FIGURES/queue_length.pdf", replace

SystemError: 
. ****************************
. * Queue Length Over Time
. ****************************
. 
. import delimited using "$PROCESSED_DATA/queue_daily.csv", varnames(1) clear
file processed_data/queue_daily.csv not found
r(601);
r(601);


In [ ]:
%%stata
****************************
* Queue stETH Over Time
****************************

import delimited using "$PROCESSED_DATA/queue_daily.csv", varnames(1) clear
gen date2 = date(date, "YMD")
format date2 %td
gen queue_steth_k = queue_steth / 1000

twoway ///
    (line queue_steth_k date2, lcolor(orange) lwidth(medthick)) ///
    , ///
    xtitle("") ///
    ytitle("stETH (thousands)") ///
    title("Lido Withdrawal Queue – stETH Amount") ///
    xlabel(, format(%tdMon-CCYY) angle(45)) ///
    graphregion(color(white)) ///
    plotregion(margin(zero))

graph export "$FIGURES/queue_steth.pdf", replace

In [ ]:
%%stata
****************************
* Wait Time Distribution (histogram)
****************************

import delimited using "$PROCESSED_DATA/queue_requests.csv", varnames(1) clear
keep if is_finalized == 1

histogram wait_days, ///
    width(0.5) ///
    fcolor(navy%70) lcolor(white) lwidth(vthin) ///
    xtitle("Days from Request to Finalization") ///
    ytitle("Density") ///
    title("Redemption Wait Time Distribution") ///
    graphregion(color(white))

graph export "$FIGURES/wait_time_hist.pdf", replace

In [ ]:
%%stata
****************************
* Monthly Median Wait Time
****************************

import delimited using "$PROCESSED_DATA/queue_requests.csv", varnames(1) clear
keep if is_finalized == 1
gen submit_date2 = date(submit_date, "YMD")
format submit_date2 %td
gen month = mofd(submit_date2)
format month %tm

collapse (median) med_wait=wait_days (p25) p25_wait=wait_days (p75) p75_wait=wait_days, by(month)

twoway ///
    (rarea p25_wait p75_wait month, color(navy%25) lcolor(navy%25)) ///
    (line  med_wait month, lcolor(navy) lwidth(medthick)) ///
    , ///
    xtitle("") ///
    ytitle("Days") ///
    title("Monthly Median Redemption Wait Time") ///
    subtitle("Shaded band: 25th–75th percentile") ///
    xlabel(, format(%tmMon-CCYY) angle(45)) ///
    legend(off) ///
    graphregion(color(white))

graph export "$FIGURES/wait_time_monthly.pdf", replace